In [1]:
import numpy as np
import math

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ============================================================
# WEEK 12 — FUNCTION 2 (PCA-AWARE GP-BASED LOCAL MAXIMISATION)
# What changed vs Week 11 (Module 23: PCA lens):
#  - Fits PCA on standardized X to learn dominant variation directions
#  - Candidate generation happens in PCA space (structured exploration)
#     (a) global exploration (raw space)
#     (b) local exploitation around x_best in PCA space (anisotropic ellipse)
#     (c) "PC-line probe": samples along the principal direction through x_best
#     (d) optional light jitter around best-known region in PCA space
#  - Novelty and min-distance measured in PCA space (remove redundancy)
#  - Adds a small PC-guidance term: move along PC most correlated with y
#  - Keeps EI + UCB dominance, but reduces random scatter
#  - Deterministic seed for reproducibility
#  - Output x_next rounded to <= 6 decimals
# ============================================================

# ----------------------------
# 1) INPUT DATA (your history)
# ----------------------------
X = np.array([
    [0.66579958, 0.12396913],
    [0.87779099, 0.77862750],
    [0.14269907, 0.34900513],
    [0.84527543, 0.71112027],
    [0.45464714, 0.29045518],
    [0.57771284, 0.77197318],
    [0.43816606, 0.68501826],
    [0.34174959, 0.02869772],
    [0.33864816, 0.21386725],
    [0.70263656, 0.92656420],
    [0.92656400, 1.02656400],   # will be clipped to 1.0
    [0.74750400, 0.20897000],
    [0.68340600, 0.06376900],
    [0.58313700, 0.01254900],
    [0.99145700, 0.00174400],
    [0.99145700, 0.00174400],   # duplicate
    [0.99947700, 0.02152800],
    [0.00757800, 0.97735900],
    [1.00000000, 0.98231600],
    [0.69158300, 0.48913100],
    [0.68265500, 0.34268700],
], dtype=float)

y = np.array([
    0.53899612, 0.42058624, -0.06562362, 0.29399291, 0.21496451,
    0.02310555, 0.24461934, 0.03874902, -0.01385762, 0.61120522,
    -0.04199554, 0.28033031, 0.62973064, 0.06695726,
    0.11670354823827367, 0.11353912028668156, 0.02426575623315439,
    0.15170334833240093, 0.003163976344064868, 0.4846853396945425,
    0.6052619631635735
], dtype=float)

# Bounds enforcement for black-box
X = np.clip(X, 0.0, 1.0)

# -------------------------------------------
# 2) DEDUPE (average y for identical points)
# -------------------------------------------
def dedupe_average(X, y):
    b = np.ascontiguousarray(X).view(
        np.dtype((np.void, X.dtype.itemsize * X.shape[1]))
    )
    _, inv = np.unique(b, return_inverse=True)
    Xu, yu = [], []
    for i in np.unique(inv):
        idx = np.where(inv == i)[0]
        Xu.append(X[idx[0]])
        yu.append(y[idx].mean())
    return np.array(Xu), np.array(yu)

X, y = dedupe_average(X, y)

# -------------------------------------------
# 3) GP model (interpretable uncertainty)
# -------------------------------------------
def make_gp(seed=2026):
    kernel = ConstantKernel(1.0, (1e-2, 1e2)) * Matern(
        length_scale=np.ones(X.shape[1]),
        length_scale_bounds=(1e-2, 1e1),
        nu=2.5
    ) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=12,
        random_state=seed
    )

def gp_cv_mse(X, y, seed=2026):
    kf = KFold(n_splits=min(5, len(y)), shuffle=True, random_state=seed)
    mses = []
    for tr, te in kf.split(X):
        gp = make_gp(seed=seed)
        gp.fit(X[tr], y[tr])
        mu, _ = gp.predict(X[te], return_std=True)
        mses.append(mean_squared_error(y[te], mu))
    return float(np.mean(mses))

cv_mse = gp_cv_mse(X, y, seed=2026)

gp = make_gp(seed=2026)
gp.fit(X, y)

# -------------------------------------------
# 4) Acquisition functions (transparent)
# -------------------------------------------
def normal_pdf(z):
    return np.exp(-0.5 * z * z) / np.sqrt(2.0 * np.pi)

def normal_cdf(z):
    return 0.5 * (1.0 + np.vectorize(math.erf)(z / np.sqrt(2.0)))

def expected_improvement(mu, std, y_best, xi=0.001):
    std = np.maximum(std, 1e-12)
    z = (mu - y_best - xi) / std
    return (mu - y_best - xi) * normal_cdf(z) + std * normal_pdf(z)

# -------------------------------------------
# 5) PCA lens (Module 23)
#    Learn principal directions of sampled X
# -------------------------------------------
scaler = StandardScaler()
Xz = scaler.fit_transform(X)

pca = PCA(n_components=2, random_state=2026)
Xp = pca.fit_transform(Xz)  # PCA coordinates
evr = pca.explained_variance_ratio_  # variance explained per PC

# Identify which PC correlates most with y (guidance direction)
# (This is a "PCA insight": which axis drives outcome variation most.)
corrs = []
for j in range(2):
    v = Xp[:, j]
    if np.std(v) < 1e-12:
        corrs.append(0.0)
    else:
        corrs.append(float(np.corrcoef(v, y)[0, 1]))
pc_star = int(np.argmax(np.abs(corrs)))  # PC index with strongest |corr|
pc_star_sign = 1.0 if corrs[pc_star] >= 0 else -1.0

# -------------------------------------------
# 6) Candidate generation (more structured)
#    Generate in PCA space to reduce randomness
# -------------------------------------------
rng = np.random.default_rng(2026)

idx_best = int(np.argmax(y))
x_best = X[idx_best]
y_best = float(y[idx_best])
n = len(y)

# Convert x_best to PCA coords
x_best_z = scaler.transform(x_best.reshape(1, -1))
x_best_p = pca.transform(x_best_z).reshape(-1)  # shape (2,)

# Trust region scale shrinks with n (structured exploitation)
# Use anisotropic scaling based on explained variance (PCA idea)
base = float(np.clip(0.95 / np.sqrt(max(n, 1)), 0.18, 0.35))  # in PCA units
pc_scales = np.array([
    base * np.sqrt(max(evr[0], 1e-6)) / np.sqrt(max(evr.mean(), 1e-6)),
    base * np.sqrt(max(evr[1], 1e-6)) / np.sqrt(max(evr.mean(), 1e-6)),
], dtype=float)
pc_scales = np.clip(pc_scales, 0.08, 0.38)

# Candidate budgets (more efficient than Week 11 huge pools)
N_global   = 15000
N_local_p  = 22000
N_line     = 12000
N_jitter_p = 8000

# (a) Global exploration in raw space
X_global = rng.uniform(0.0, 1.0, size=(N_global, 2))

# Helper: PCA-space samples -> raw-space
def pca_to_raw(Xp_samples):
    Xz_samples = pca.inverse_transform(Xp_samples)
    Xraw = scaler.inverse_transform(Xz_samples)
    return np.clip(Xraw, 0.0, 1.0)

# (b) Local anisotropic ellipse around x_best in PCA space
Xp_local = rng.normal(loc=x_best_p, scale=pc_scales, size=(N_local_p, 2))
X_local = pca_to_raw(Xp_local)

# (c) PC-line probe: move along dominant PC through x_best
# This is "principal direction exploration" rather than random scatter.
t = rng.normal(loc=0.0, scale=1.0, size=(N_line, 1))
line_dir = np.zeros((1, 2))
line_dir[0, pc_star] = 1.0
Xp_line = x_best_p + (t * (pc_scales[pc_star] * 2.2)) * line_dir
# small perpendicular jitter to avoid collapsing to one line
perp_dir = np.zeros((1, 2))
perp_dir[0, 1 - pc_star] = 1.0
Xp_line = Xp_line + rng.normal(0.0, pc_scales[1 - pc_star] * 0.35, size=(N_line, 1)) * perp_dir
X_line = pca_to_raw(Xp_line)

# (d) Light PCA jitter around the best-y quartile centroid (reduces redundancy)
q = np.quantile(y, 0.75)
top_idx = np.where(y >= q)[0]
Xp_top_centroid = Xp[top_idx].mean(axis=0) if len(top_idx) > 0 else x_best_p
Xp_jitter = rng.normal(loc=Xp_top_centroid, scale=pc_scales * 0.75, size=(N_jitter_p, 2))
X_jitter = pca_to_raw(Xp_jitter)

Xcand = np.vstack([X_global, X_local, X_line, X_jitter])

# -------------------------------------------
# 7) Score candidates: EI + UCB + PCA-novelty + PC-guidance
# -------------------------------------------
mu, std = gp.predict(Xcand, return_std=True)

xi = 0.001
ei = expected_improvement(mu, std, y_best, xi=xi)

kappa = 2.0
ucb = mu + kappa * std

# Novelty & min-distance in PCA space (PCA removes redundancy)
Xcand_z = scaler.transform(Xcand)
Xcand_p = pca.transform(Xcand_z)

# PCA-space distances to existing samples
dists_p = np.sqrt(((Xcand_p[:, None, :] - Xp[None, :, :]) ** 2).sum(axis=2))
min_dist_p = dists_p.min(axis=1)

# Separation threshold in PCA space
min_sep_p = float(np.clip(0.90 / np.sqrt(max(n, 1)), 0.12, 0.22))
valid = min_dist_p >= min_sep_p

def zscore(v):
    return (v - v.mean()) / (v.std() + 1e-12)

ei_z  = zscore(ei)
ucb_z = zscore(ucb)
nov_z = zscore(min_dist_p)

# PC-guidance: encourage movement in the y-improving PC direction
# (signed coordinate along the most y-correlated PC)
pc_coord = pc_star_sign * Xcand_p[:, pc_star]
pc_guide = zscore(pc_coord)

# Week 12 mix (PCA-aligned):
#  - EI dominates (exploit)
#  - UCB keeps uncertainty-aware exploration
#  - PCA-novelty avoids redundant directions
#  - PC-guidance focuses on the principal direction that historically improves y
score = 0.58 * ei_z + 0.24 * ucb_z + 0.10 * nov_z + 0.08 * pc_guide

score_masked = np.where(valid, score, -np.inf)
best_idx = int(np.argmax(score_masked))

x_next = np.round(Xcand[best_idx], 6)

# -------------------------------------------
# 8) Transparency prints
# -------------------------------------------
topk = 5
top_idx = np.argsort(score_masked)[-topk:][::-1]

print("================================================")
print("WEEK 12 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)")
print("================================================")
print(f"GP kernel learned: {gp.kernel_}")
print(f"CV MSE (sanity check): {cv_mse:.6f}")
print(f"x_best = [{x_best[0]:.6f}, {x_best[1]:.6f}], y_best = {y_best:.6f}")
print("------------------------------------------------")
print("PCA summary (on standardized X):")
print(f"  explained_variance_ratio = [{evr[0]:.6f}, {evr[1]:.6f}]")
print(f"  corr(PC1,y)={corrs[0]:.6f}  corr(PC2,y)={corrs[1]:.6f}")
print(f"  pc_star = PC{pc_star+1} (sign={pc_star_sign:+.0f})")
print(f"  pc_scales = [{pc_scales[0]:.6f}, {pc_scales[1]:.6f}]")
print("------------------------------------------------")
print(f"min_sep_p (PCA-space) = {min_sep_p:.6f}")
print("------------------------------------------------")
print(f"x_next = [{x_next[0]:.6f}, {x_next[1]:.6f}]")
print(f"mu(x_next)          = {mu[best_idx]:.6f}")
print(f"std(x_next)         = {std[best_idx]:.6f}")
print(f"EI(x_next)          = {ei[best_idx]:.6f}")
print(f"UCB(x_next)         = {ucb[best_idx]:.6f}")
print(f"min_dist_p(x_next)  = {min_dist_p[best_idx]:.6f}")
print(f"pc_guide(x_next)    = {pc_guide[best_idx]:.6f}")
print("score_parts = EI_w=0.58, UCB_w=0.24, nov_w=0.10, pc_w=0.08")
print("------------------------------------------------")
print("Top candidates (for transparency):")
for rank, i in enumerate(top_idx, start=1):
    x = Xcand[i]
    print(
        f"{rank}) x=[{x[0]:.6f},{x[1]:.6f}]  "
        f"mu={mu[i]:.6f} std={std[i]:.6f} EI={ei[i]:.6f} "
        f"UCB={ucb[i]:.6f} minDistP={min_dist_p[i]:.6f} "
        f"pcGuide={pc_guide[i]:.6f} score={score_masked[i]:.6f}"
    )


C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:442: ConvergenceWarning: The optimal value found for

WEEK 12 FUNCTION 2 — NEXT DATA POINT (<=6 DECIMALS)
GP kernel learned: 0.874**2 * Matern(length_scale=[0.0168, 10], nu=2.5) + WhiteKernel(noise_level=0.00756)
CV MSE (sanity check): 0.027907
x_best = [0.683406, 0.063769], y_best = 0.629731
------------------------------------------------
PCA summary (on standardized X):
  explained_variance_ratio = [0.511674, 0.488326]
  corr(PC1,y)=0.106674  corr(PC2,y)=0.134681
  pc_star = PC2 (sign=+1)
  pc_scales = [0.214892, 0.209932]
------------------------------------------------
min_sep_p (PCA-space) = 0.201246
------------------------------------------------
x_next = [0.711209, 0.665305]
mu(x_next)          = 0.592397
std(x_next)         = 0.106180
EI(x_next)          = 0.025924
UCB(x_next)         = 0.804756
min_dist_p(x_next)  = 0.494095
pc_guide(x_next)    = -0.851520
score_parts = EI_w=0.58, UCB_w=0.24, nov_w=0.10, pc_w=0.08
------------------------------------------------
Top candidates (for transparency):
1) x=[0.711209,0.665305]  mu=0.

C:\Anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k1__k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
